In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-ingestion") \
    .getOrCreate()

print("Spark Running:", spark.version)

Spark Running: 3.5.0


In [2]:
import os
import json
import time
import requests
from datetime import datetime

from pyspark.sql.functions import current_timestamp, lit

In [3]:
API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

BASE_PATH = "/home/jovyan/work/output/bronze"
MANIFEST_PATH = f"{BASE_PATH}/manifest.json"
ERROR_LOG_PATH = f"{BASE_PATH}/error_log.txt"

In [4]:
os.makedirs(BASE_PATH, exist_ok=True)

endpoints = [
    "orders", "order_items", "customers",
    "products", "sellers", "payments"
]

for ep in endpoints:
    os.makedirs(f"{BASE_PATH}/{ep}", exist_ok=True)

print("Folders ready")

Folders ready


In [5]:
def get_token():
    response = requests.post(
        f"{API_BASE_URL}/api/v1/auth/token",
        json={
            "username": API_USERNAME,
            "password": API_PASSWORD
        }
    )
    return response.json()["access_token"]

In [17]:
def load_manifest():
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            return json.load(f)
    return {}

def save_manifest(manifest):
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)

In [18]:
def log_error(msg):
    with open(ERROR_LOG_PATH, "a") as f:
        f.write(f"{datetime.now()} - {msg}\n")

In [19]:
def make_request(url, headers, params):
    for i in range(5):
        response = requests.get(url, headers=headers, params=params)

        if response.status_code == 200:
            return response

        elif response.status_code == 401:
            print("Refreshing token...")
            token = get_token()
            headers["Authorization"] = f"Bearer {token}"

        elif response.status_code in [500, 429]:
            wait = 2 ** i
            print(f"Retry after {wait}s")
            time.sleep(wait)

        else:
            log_error(response.text)
            break

    raise Exception("API failed")

In [20]:
def fetch_data(endpoint, token):
    headers = {"Authorization": f"Bearer {token}"}
    manifest = load_manifest()

    processed_pages = manifest.get(endpoint, [])
    last_date = manifest.get(f"{endpoint}_date", "2018-07-01")

    all_data = []
    page = 1

    while True:
        params = {
            "page": page,
            "page_size": 1000
        }

        # Date filter only for orders & order_items
        if endpoint in ["orders", "order_items"]:
            params["date_from"] = last_date

        # Skip already processed pages
        if page in processed_pages:
            print(f"Skipping {endpoint} page {page}")
            page += 1
            continue

        url = f"{API_BASE_URL}/api/v1/data/{endpoint}"

        response = make_request(url, headers, params)
        json_data = response.json()

        records = json_data.get("data", [])

        if not records:
            break

        valid_records = []
        invalid_records = []

        for r in records:
            if isinstance(r, dict):
                valid_records.append(r)
            else:
                invalid_records.append(r)

        all_data.extend(valid_records)

        # Log bad records
        if invalid_records:
            log_error(f"{endpoint} page {page} has bad records")

        print(f"{endpoint} page {page} fetched")

        # FIX: Add page only if not already present
        if page not in processed_pages:
            processed_pages.append(page)

        page += 1

    # Save manifest
    manifest[endpoint] = processed_pages

    # Update date for incremental load
    if endpoint in ["orders", "order_items"]:
        manifest[f"{endpoint}_date"] = datetime.now().strftime("%Y-%m-%d")

    save_manifest(manifest)

    return all_data

In [21]:
def save_to_bronze(data, endpoint):
    if not data:
        print(f"No data for {endpoint}")
        return

    df = spark.createDataFrame(data)

    df = df.withColumn("_ingested_at", current_timestamp()) \
           .withColumn("_source_endpoint", lit(endpoint))

    df.coalesce(1).write \
        .mode("append") \
        .option("header", True) \
        .csv(f"{BASE_PATH}/{endpoint}")

    print(f"{endpoint} saved")

In [22]:
def run_ingestion():
    token = get_token()

    for ep in endpoints:
        print(f"\nProcessing {ep}...")

        try:
            data = fetch_data(ep, token)
            save_to_bronze(data, ep)

        except Exception as e:
            log_error(f"{ep} failed: {str(e)}")
            print(f"{ep} failed")

In [23]:
run_ingestion()


Processing orders...
Skipping orders page 1
Skipping orders page 2
Skipping orders page 3
Skipping orders page 4
Skipping orders page 5
Skipping orders page 6
Skipping orders page 7
Skipping orders page 8
Skipping orders page 9
Skipping orders page 10
Skipping orders page 11
Skipping orders page 12
Skipping orders page 13
No data for orders

Processing order_items...
Skipping order_items page 1
Skipping order_items page 2
Skipping order_items page 3
Skipping order_items page 4
Skipping order_items page 5
Skipping order_items page 6
Skipping order_items page 7
Skipping order_items page 8
Skipping order_items page 9
Skipping order_items page 10
Skipping order_items page 11
Skipping order_items page 12
Skipping order_items page 13
Skipping order_items page 14
Skipping order_items page 15
Skipping order_items page 16
No data for order_items

Processing customers...
Skipping customers page 1
Skipping customers page 2
Skipping customers page 3
Skipping customers page 4
Skipping customers pa